# From model to practice

`03-model-results` and `04-schedule-6-deep-dive` establish that a model beats
every classical alternative, and by how much. Neither asks the practical
questions a regulator, funder or charity would ask next: is a score like this
actually useful, how would someone put it to work, and what would make it
better? This notebook is deliberately less quantitative than the rest of the
series — no cohort is fitted or scored here — because the questions it takes
on are about deployment and data, not about model selection.

Three things follow, in order:

1. why a data-driven vulnerability measure is worth having at all;
2. two ways a model like this one could actually run — scoring T3010 filings
   as they arrive, and scoring organisations that submit their own numbers
   voluntarily;
3. what this study would need — data it does not have — to go further.

## 1. Why a data-driven vulnerability measure is useful

A charity regulator's basic problem is scale without proportional capacity: a
registry of tens of thousands of filers, an annual return from each, and a
review process that cannot read all of them closely. Left unaided, oversight
becomes either broad and shallow (spot checks, or waiting for a public
complaint) or narrow and reactive (auditing whoever already tripped a
bright-line rule, such as missing the disbursement quota — see
`04-schedule-6-deep-dive`'s disbursement-quota analysis for how weak a signal
that particular trigger turns out to be on its own). Neither targets
attention at the organisations most likely to actually fail.

A predictive score is a way to allocate a fixed review budget better than
either of those defaults, and its value is exactly the shape this project has
been measuring throughout: at a 5% review budget, the best model here catches
about 41% of the following year's deregistrations, against the 5% a random
sample would catch (`03-model-results`). That gap — lift — is the entire case
for a data-driven measure over a rule of thumb or a complaint-driven one.

**It is a case for triage, not for punishment**, and the distinction matters
because of who gets flagged. Exits concentrate among small, young
organisations (`01-data-and-outcomes`, "who exits"), and a screening tool used
punitively would systematically fall on exactly the charities least able to
absorb an audit or a funding freeze — often the newest, most under-resourced,
most community-rooted ones. Used instead to route outreach, technical
assistance or a lighter-touch check-in ahead of trouble, the same score
targets help rather than harm at the same population. Which use a regulator
chooses is a policy decision this project cannot make, but the difference
between them is the difference between a genuinely useful instrument and a
harmful one built from the same numbers.

**The audience is wider than the regulator.** A validated, well-calibrated
score is also useful to:

* **funders and donors**, as a substitute for the overhead-ratio heuristics
  that dominate consumer-facing charity ratings today — and this project's
  own results are a direct argument against those heuristics:
  `02-features-and-benchmarks` and `03-model-results` both find the
  administrative cost ratio pointing the *wrong* way for predicting exit,
  which is precisely the ratio many donor-facing rating tools still lean on
  hardest;
* **umbrella and sector bodies**, for sector-wide health monitoring — is
  deregistration concentrated in one province, one charitable category, one
  funding model, this year;
* **the organisations themselves**, as a governance signal a board can act on
  before a funder or regulator ever sees a number;
* **researchers and policymakers**, as an evidence base for questions about
  the sector that a spreadsheet of totals cannot answer on its own.

None of this changes what the score actually is. It predicts
*deregistration*, which bundles collapse together with solvent wind-ups and
amalgamations (`03-model-results`, "what this does and does not establish"),
and every use above should be built around that limit rather than around the
more dramatic claim — "this charity is failing" — that the number invites but
does not support.

## 2. Putting a model like this one to work

Two deployment shapes cover most of what a model like this could actually do.
They differ in where the data comes from, not in what happens to it once it
arrives — both route into the same kind of ranked list
`charity_risk.pipeline.run_watchlist` already produces for the 2022 cohort in
`03-model-results`.

### 2a. Scoring T3010 filings as they arrive

This is the deployment the project already demonstrates, essentially
unchanged. Each year's newly filed cohort has features but, until enough
later years exist to confirm an absence is permanent, no trustworthy label —
exactly the situation `charity_risk.dataset.scoring_frame` and the 2022
watchlist in `03-model-results` are built for. Extending that watchlist
forward every year is mostly an operations problem, not a modelling one.

**It does not require the regulator's participation.** The T3010 extract this
whole project runs on is public. Anything built here could equally be run,
and published, by an umbrella body, a university research centre, or a
sector watchdog with no more access than this repository already has — a
different governance model from a regulator-run system, with different
incentives and different trust properties, discussed further below.

**If the regulator does run it**, the natural integration point is upstream
of a human decision, not in place of one: a ranked list feeding a compliance
officer's discretion about which files to open, or a trigger for a
proportionate, non-punitive check-in letter rather than an audit.

Three operational details follow directly from how this project is built.

**The training data is always stale by construction.** The exit label needs
at least `MIN_LOOKAHEAD_YEARS` (two) confirmed years after the filing year
before it can be trusted (`charity_risk.outcomes`), so the freshest year a
model can be *trained* on always trails the freshest year it can *score* by
that much. A production system would refit annually on a rolling window and
treat this lag as a structural property, not a bug to fix.

**Calibration drift needs watching, not just discrimination.** A ranking that
still sorts risk correctly can drift out of calibration — a slope and
intercept that used to sit near (1, 0) can wander as the underlying
population or economy changes (2020-2021 are pandemic years, and this project
says so plainly; a later cohort is a different economy again). The ensemble
model's near-perfect calibration (slope 0.95, intercept 0.04 — see
`03-model-results`) is what makes "expected number of exits in this year's
filing batch" a defensible sentence to say out loud; that property should be
checked every year the model is refit, not assumed to persist.

**Explainability is an operational requirement here in a way it was not a
research one.** A public regulator using a score to justify attention needs
to be able to say why an organisation was flagged, both to the organisation
and, eventually, to whoever reviews the regulator's own conduct. Gradient
boosting and the ensemble built from it are the two best-ranked models in
this project and also the two hardest to explain in a sentence, while the
individual ratios in the classical logits, or a permutation-importance table
like the one in `03-model-results`, translate directly into "here is what was
unusual about this filing." The HS risk index in `06-hs-risk-index` is built
for exactly this: every score decomposes into five dimension scores and
twelve percentiles against last year's sector, so the reason for a flag can
be read off the score itself, with no model to explain. A regulator
constrained to publish its
methodology may reasonably trade some ranking performance for that property —
the mirror image of what `04-schedule-6-deep-dive` finds from a completely
different direction: the detailed return buys the most accuracy precisely for
the simple, explainable models a transparency-constrained institution would
actually be allowed to run.

### 2b. Voluntary submission against the T3010-trained model

The second shape inverts who supplies the data. An organisation — one not yet
required to file the detailed return, one outside CRA's charitable register
entirely, or simply one that wants a private read before its own board sees
the number — submits its financials directly, and the model trained on T3010
filings scores them.

**How much does an organisation actually have to hand over?** This project's
feature groups already answer that, because they were built to keep
short-form and long-form filers comparable in the first place
(`charity_risk.features`). Group the same feature set by the input burden
each group implies, from cheapest to most demanding, counting only features
not already covered by a cheaper tier:

In [1]:
from charity_risk.features import feature_columns

# Tiers by how much an organisation would have to supply to compute them, not
# by topic — mirrors the "core" vs "Schedule 6" split the rest of this
# project already draws, with the core split one step further.
tiers = {
    "headline totals + age": ("tuckman_chang", "trussel", "structure"),
    "+ revenue mix, one prior year": ("composition", "dynamics"),
    "+ itemised balance sheet & cost structure": ("liquidity", "schedule_6"),
}

seen = set()
rows = []
for label, groups in tiers.items():
    new_columns = [c for c in feature_columns(list(groups)) if c not in seen]
    seen.update(new_columns)
    rows.append({"tier": label, "new_features": len(new_columns),
                 "cumulative_features": len(seen)})

import pandas as pd
pd.DataFrame(rows).set_index("tier")

,new_features,cumulative_features
tier,,
headline totals + age,17,17
"+ revenue mix, one prior year",20,37
+ itemised balance sheet & cost structure,57,94


Seventeen of the ninety-four features across all three tiers (18%) come from
nothing more than five totals almost any organisation already tracks for its
own books — revenue, expenditure, assets, liabilities, net assets — plus its
age: the whole of the Tuckman-Chang, Trussel and `structure` groups, the same
ratios `02-features-and-benchmarks` reproduces from the classical literature.
A second tier — twenty more features, 37 of 94 (39%) cumulatively — asks for
revenue broken down by source and one prior year of the same headline
totals, enough to compute growth rates and a revenue-concentration index.
The remaining fifty-seven, well over half the set, need the itemised
Schedule 6 balance sheet and cost structure `04-schedule-6-deep-dive` spends
a whole notebook examining.

That tiering is a design, not just an observation: an intake form could ask
for the cheap tier first and return an immediate, coarser score, then offer a
sharper one if the organisation is willing to supply the rest — the same "how
much does more detail buy" question `04-schedule-6-deep-dive` already
answered for the T3010 register, reframed as a design choice for a
self-assessment tool instead of a research question about it. And a similar
answer likely applies here too: a flexible model built from the cheap tier
alone should recover most of a fuller model's ranking performance, the same
way gradient boosting needed the itemised return far less than the
elastic-net logit did on the T3010 register itself.

The HS risk index (`06-hs-risk-index`) suits the same staged design. Its
solvency and profitability ratios need only headline totals, its revenue
dimension needs the revenue mix and prior years, and its liquidity dimension
needs the itemised balance sheet. Because it already scores only the
dimensions an organisation has data for, a partial submission yields a
partial but still readable index rather than an imputed one.

Four problems are specific to this channel, and none of them go away by
having a good model.

**Who submits is not who files.** The model is trained on the population of
organisations that actually registered and filed a T3010 — already a
selected group. Voluntary submitters are a further, differently selected
group: possibly organisations already worried enough to seek reassurance
(adverse selection, making the pool riskier than the training population),
possibly the opposite — confident organisations showcasing themselves to a
funder. Either way, nothing guarantees the submitting population resembles
the training one, and the honest response is to monitor it rather than
assume it away: refit the calibration layer
(`charity_risk.evaluate.score_metrics`'s calibration slope and intercept
already exist for exactly this) separately for the voluntary channel once
enough submissions accumulate to check it.

**Self-reported numbers here are unaudited in a different sense than T3010
figures are.** Even a T3010 return is self-reported by most filers — CRA does
not verify every line — but it is filed under a registration a charity can
lose, which is itself a discipline this project's whole label depends on. A
private submission carries no equivalent consequence for a misstated number.
A production intake form should build in what it can: consistency checks an
accounting identity implies (does the change in reported net assets roughly
match reported revenue minus expenditure), and should not pretend a
self-report is a filed return.

**Accounting convention needs to travel, not just the numbers.** Two
conventions this project spends real effort getting right —
`fields.is_collected`'s distinction between "asked and left blank" and "never
asked", and the cash-versus-accrual basis flagged by `is_accrual` — are
exactly the kind of thing an organisation unfamiliar with T3010 reporting
could get wrong in a self-report without anyone noticing. A voluntary-intake
version of the same distinctions, stated plainly on the form rather than
inferred from a raw filing, is not optional.

**Who runs the service changes what gets submitted, honestly, at all.** A
charity has little reason to give its own regulator a self-diagnosed risk
score. An independent host — a sector umbrella body, a university research
centre, a philanthropic intermediary — with a clear, credible commitment not
to forward individual results is far more likely to get candid numbers, for
the same reason credit counselling is typically offered by an independent
non-profit rather than by a lender. The value of the score to the submitting
organisation — a governance signal, evidence for its own board, a benchmark
against the sector — does not require the score to ever leave the
organisation's hands.

## 3. What this study needs to go further

Every caveat in `03-model-results` and `04-schedule-6-deep-dive` points at
data this project does not have. Six extensions, roughly in order of how much
each would change the answer:

**Linked revocation reasons**, already named as the single highest-value
extension in `03-model-results`: separating collapse from solvent wind-up,
amalgamation and administrative revocation would turn `exit_next_year` from a
proxy into something closer to an actual failure label, and would let every
result in this project be re-asked against the outcome it is really trying to
predict.

**A longer panel.** Five years, with a two-year lookahead requirement, leaves
exactly one clean out-of-time comparison (`03-model-results`, "not
established"). More years would support real duration modelling — a hazard
that varies with how long a charity has been at risk, not just whether it is
— and would make Alam, Gao and Jones's actual contribution testable here for
the first time: their Grassmann-manifold encoding of a firm's full history
needs a training panel longer than the two annual observations this window
provides (`03-model-results`, "what this does and does not establish").

**Signal with the immediacy the T3010 structurally lacks.** Alam et al.'s
deep model has an edge this project's cannot reproduce partly because their
features include market prices that move daily; a charity has no share
price, but donation-platform activity, media mentions, job postings and
search interest are all closer analogues than another annual financial ratio
would be, and none of them are in this project.

**Governance data.** T3010 covers more ground than the two files this
project reads — `01-data-and-outcomes` notes that Section A, B and C and
Schedules 1 and 5 exist for 2019-2022 and were excluded only because the 2023
extract drops them. Restoring them, even at the cost of the 2023 lookahead
year, would let board composition, related-party transactions and executive
turnover be tested as leading indicators the way the corporate-failure
literature already treats them, rather than left out entirely.

**Provincial and macroeconomic context.** Charitable activity is also
regulated provincially, and none of that record is linked here; and the
training cohort's two years (2020-2021) are pandemic years by construction
(`03-model-results`), with no macro control to separate an economy-wide shock
from organisation-specific weakness. Both are missing context this project is
explicit about lacking rather than pretending doesn't matter.

**A feedback loop.** Nothing here, or plausibly available anywhere yet,
supports the question that would matter most to a regulator deciding what to
do with a score: does an early check-in actually change an at-risk charity's
trajectory? Answering that needs data on past interventions and their
outcomes, which does not exist yet because the interventions this project
would inform do not exist yet either. It is the natural last step, and
necessarily downstream of everything else here.

## Closing

None of these six are required to make the case in section 1. A validated,
well-calibrated model built only from data CRA already collects and already
publishes beats every classical alternative tried against it
(`03-model-results`) — the statistical case for a data-driven measure is
already made. What remains is not better statistics so much as better
plumbing: who runs it, who can see a result, what happens to an organisation
once it is flagged, and how much an organisation with nothing to hide from
CRA can be trusted to hand over voluntarily. Those are not modelling
questions, and this project does not resolve them. Naming them plainly, and
tying each one to a specific result rather than a general worry, is what this
notebook set out to do.